# Module CLI Runner (Notebook)

Use this notebook to quickly test the standalone Team Assistant and Workflow modules without API integration.

Run cells top-to-bottom.

In [ ]:
from pathlib import Path
import sys

# Ensure repo root is importable when notebook runs from workspace root.
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from backend.app.services.team.service import (
    summarize_content,
    extract_tasks,
    run_deadline_reminders,
)
from backend.app.services.workflow.service import (
    create_workflow_item,
    move_stage,
    create_milestone,
    update_milestone,
)

print("Imports loaded.")

ModuleNotFoundError: No module named 'pydantic'

In [ ]:
sample_content = """
Meeting notes:
We discussed next week's content pieces and owners.
Action: Shamik prepare trend shortlist tomorrow
TODO: Batu finalize storyboard ASAP
- Shufen review draft script by Friday
""".strip()

summary = summarize_content(sample_content, source_type="meeting", title="Weekly planning")
print("=== Summary ===")
print(summary.model_dump())

tasks = extract_tasks(
    sample_content,
    owner_candidates=["Shamik", "Batu", "Shufen"],
    default_due_days=3,
)

print("\n=== Extracted Tasks ===")
for task in tasks:
    print(task.model_dump())

In [ ]:
reminders = run_deadline_reminders(tasks, window_hours=24)

print("=== Deadline Reminders ===")
for reminder in reminders:
    print(reminder.model_dump())

if not reminders:
    print("No reminders generated for current due dates.")

In [ ]:
item = create_workflow_item(
    title="Budget Mythbuster Short",
    description="Draft and publish one short-form video",
    owner="Batu",
)

print("=== Workflow Item (Initial) ===")
print(item.model_dump())

item = move_stage(item, "Brief", note="Concept approved")
item = move_stage(item, "Production", note="Brief signed off")

print("\n=== Workflow Item (After Stage Moves) ===")
print(item.model_dump())

milestone = create_milestone(
    item_id=item.item_id,
    title="Script lock",
    owner="Shufen",
    criteria=["Hook approved", "CTA finalized"],
)

milestone = update_milestone(
    milestone,
    status="in_progress",
    notes="Waiting for final CTA wording.",
)

print("\n=== Milestone ===")
print(milestone.model_dump())

In [ ]:
# Optional: quick edge-case checks
try:
    invalid = move_stage(item, "Publish", note="Attempt to skip review")
    print("Unexpected success:", invalid.model_dump())
except Exception as exc:
    print("Expected transition validation error:", exc)